In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install xgboost scikit-learn pandas numpy matplotlib scipy hurst -q

In [3]:
!cp "/content/drive/MyDrive/Thesis/bitbrains/bitbrains_xval.py" /content/bitbrains_xval.py

In [4]:
!python bitbrains_xval.py --stage all \
    --data_dir "/content/drive/MyDrive/Thesis/bitbrains/fastStorage/2013-8" \
    --output_dir "/content/drive/MyDrive/Thesis/bitbrains/results" \
    --reset_checkpoint

found 1250 CSV files in /content/drive/MyDrive/Thesis/bitbrains/fastStorage/2013-8
scanning — approx 3 min …
  250 / 1250
  500 / 1250
  750 / 1250
  1000 / 1250
  1250 / 1250

scanned 1240 VMs total
overall mean CPU : 7.1%
overall median CPU: 1.5%

  VMs with mean < 5%: 76.8%
  VMs with mean < 10%: 84.0%
  VMs with mean < 15%: 86.5%
  VMs with mean < 20%: 87.6%

active VMs (mean>15.0%, std>3.0%): 157 / 1240  (12.7%)

workload characterisation (active VMs, n=157):
  CV    median=1.057  mean=1.044
  Hurst median=0.023  mean=0.068
  ACF1  median=0.974  mean=0.954

active_vms saved → /content/drive/MyDrive/Thesis/bitbrains/results/active_vms.csv
old checkpoint deleted (--reset_checkpoint)
starting fresh (no checkpoint found)

training on 157 active VMs × 4 horizons
checkpoint saved every 10 VMs
  [ 10/157] 6%  results: 40  errors: 0  ✅
  [ 20/157] 13%  results: 80  errors: 0  ✅
  [ 30/157] 19%  results: 120  errors: 0  ✅
  [ 40/157] 25%  results: 160  errors: 0  ✅
  [ 50/157] 32%  results

In [5]:
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/Thesis/bitbrains/results"

per_vm     = pd.read_csv(f"{OUT_DIR}/bitbrains_per_vm.csv")
active_vms = pd.read_csv(f"{OUT_DIR}/active_vms.csv")

# normalise vm_id
per_vm["vm_id"]     = per_vm["vm_id"].apply(lambda x: str(int(float(x))))
active_vms["vm_id"] = active_vms["vm_id"].apply(lambda x: str(int(float(x))))

merged = per_vm.merge(active_vms[["vm_id","cv","hurst","acf1"]], on="vm_id", how="left")

# ── Comparable subset: CV < 0.7 (matches Alibaba predictability profile)
comparable = merged[merged["cv"] < 0.7]
print(f"Comparable VMs (CV<0.7): {comparable['vm_id'].nunique()} VMs\n")

print(f"{'Horizon':<10} {'Naive R²':>10} {'ML R²':>10} {'Δ pp':>8} {'Skill':>8} {'Win%':>8}")
print("-" * 58)
for h in ["10min","30min","60min","120min"]:
    hdf      = comparable[comparable["horizon"] == h]
    naive_r2 = hdf["naive_r2"].median()
    ml_r2    = hdf["ml_r2"].median()
    delta_pp = hdf["r2_delta"].median() * 100
    skill    = hdf["skill"].median()
    win_pct  = hdf["ml_wins"].mean() * 100
    print(f"{h:<10} {naive_r2:>10.3f} {ml_r2:>10.3f} {delta_pp:>8.2f} {skill:>8.4f} {win_pct:>7.1f}%")

# ── Alibaba reference
print("\nAlibaba hetero-ensemble (for comparison):")
ali = {"10min":(0.9188,+0.25),"30min":(0.8361,+0.43),"60min":(0.7878,+1.33),"120min":(0.7178,+4.64)}
for h,(r2,d) in ali.items():
    print(f"  {h}: naive_R²={r2:.3f}  Δpp={d:+.2f}")

Comparable VMs (CV<0.7): 15 VMs

Horizon      Naive R²      ML R²     Δ pp    Skill     Win%
----------------------------------------------------------
10min           0.588      0.691     8.76   0.0528    80.0%
30min           0.060      0.611    31.30   0.1466    86.7%
60min           0.030      0.601    28.95   0.1630    86.7%
120min         -0.202      0.456    35.96   0.2179    80.0%

Alibaba hetero-ensemble (for comparison):
  10min: naive_R²=0.919  Δpp=+0.25
  30min: naive_R²=0.836  Δpp=+0.43
  60min: naive_R²=0.788  Δpp=+1.33
  120min: naive_R²=0.718  Δpp=+4.64


In [6]:
# Hurst stratification — add after the comparable subset analysis
print("\nHurst-stratified results (all active VMs):")
print(f"{'Horizon':<10} {'Hurst<0.3':>12} {'0.3-0.5':>10} {'Hurst>0.5':>12}")
print("-" * 48)

hurst_bins  = [0, 0.3, 0.5, float("inf")]
hurst_labels = ["<0.3", "0.3-0.5", ">0.5"]
merged["hurst_bin"] = pd.cut(merged["hurst"], bins=hurst_bins, labels=hurst_labels)

for h in ["10min","30min","60min","120min"]:
    hdf = merged[merged["horizon"] == h]
    row = f"{h:<10}"
    for b in hurst_labels:
        g = hdf[hdf["hurst_bin"] == b]
        if len(g) < 3:
            row += f"  {'—':>10}"
        else:
            row += f"  {g['skill'].median():>+10.4f}"
    print(row)

print("\n(values = median skill score; positive = ML beats naive)")


Hurst-stratified results (all active VMs):
Horizon       Hurst<0.3    0.3-0.5    Hurst>0.5
------------------------------------------------
10min          -0.9714     +0.0528     -1.1481
30min          -0.7100     +0.1306     -1.6030
60min          -0.6245     +0.1938     -2.2795
120min         -0.4589     +0.3219     -2.1266

(values = median skill score; positive = ML beats naive)


In [7]:
print(merged.groupby("hurst_bin")["vm_id"].nunique())

hurst_bin
<0.3       47
0.3-0.5     9
>0.5        4
Name: vm_id, dtype: int64


/tmp/ipykernel_23244/791818355.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(merged.groupby("hurst_bin")["vm_id"].nunique())


In [8]:
# Add n per bin to the printout
for h in ["10min","30min","60min","120min"]:
    hdf = merged[merged["horizon"] == h]
    row = f"{h:<10}"
    for b in hurst_labels:
        g = hdf[hdf["hurst_bin"] == b]
        if len(g) < 5:
            row += f"  {'(n<5)':>10}"
        else:
            row += f"  {g['skill'].median():>+10.4f}"
    print(row)

10min          -0.9714     +0.0528       (n<5)
30min          -0.7100     +0.1306       (n<5)
60min          -0.6245     +0.1938       (n<5)
120min         -0.4589     +0.3219       (n<5)


In [10]:
from IPython.display import IFrame
IFrame("/content/drive/MyDrive/Thesis/bitbrains/results/bitbrains_cross_validation.pdf", width=900, height=500)